In [ ]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 33.1 MB/s eta 0:00:00


In [ ]:
import re
import pandas as pd
import spacy

# =========================
# تحميل نموذج spacy
# =========================
nlp = spacy.load("en_core_web_sm")

# =========================
# كلمات المهارات الرياضية
# =========================
skill_keywords = {
    'passing': ['passing','pass','assist','playmaking','through ball','cross','distribution'],
    'shooting': ['shooting','shoot','finishing','goal','scoring','volley','strike'],
    'dribbling': ['dribbling','dribble','ball control','take on'],
    'defending': ['defending','tackle','marking','interception','clearance','block'],
    'pace': ['speed','pace','fast','quick','sprint','acceleration'],
    'movement_acceleration': ['acceleration','burst'],
    'movement_sprint_speed': ['sprint speed','top speed'],
    'physic': ['strength','stamina','fitness','endurance','athletic'],
    'power_strength': ['strength','power','strong'],
    'power_stamina': ['stamina','endurance','work rate'],
    'power_jumping': ['jump','heading','aerial','leap'],
    'mentality_aggression': ['aggressive','competitive','determined'],
    'mentality_positioning': ['positioning','tactical awareness','reading the game'],
    'mentality_vision': ['vision','anticipate','playmaking'],
    'teamwork': ['teamwork','team player','leadership','communication'],
    'tactics': ['tactical','strategy','decision making','game intelligence']
}

# =========================
# تحديد هل CV رياضي
# =========================
def is_sports_cv(text):
    sports_keywords = [
        'football','soccer','player','striker','midfielder',
        'defender','goalkeeper','club','league','match','coach'
    ]
    text_lower = text.lower()
    return any(word in text_lower for word in sports_keywords)


# =========================
# استخراج الإنجازات
# =========================
def count_achievements(cv_text):
    """
    Counts the number of achievements mentioned in a CV text.
    Each line containing a keyword counts as one achievement, regardless of numbers.
    """
    keywords = [
        'title','trophy','award','awards','championship','cup',
        'winner','best midfielder','captain'
    ]
    total = 0
    for line in cv_text.split('\n'):
        line_lower = line.lower()
        if any(kw in line_lower for kw in keywords):
            total += 1
    return total

# =========================
# استخراج المعلومات البدنية
# =========================
def extract_physical_info(text):
    result = {'age': None, 'height_cm': None, 'weight_kg': None}

    # Age
    age_match = re.search(r'(\d{1,2})\s*years?\s*old', text, re.IGNORECASE)
    if age_match:
        result['age'] = int(age_match.group(1))

    # Height in cm
    height_cm_match = re.search(r'(\d{3})\s*cm', text, re.IGNORECASE)
    if height_cm_match:
        result['height_cm'] = int(height_cm_match.group(1))

    # Height in feet and inches (5’9”)
    height_ft_match = re.search(r"(\d)[’'](\d{1,2})", text)
    if height_ft_match:
        feet = int(height_ft_match.group(1))
        inches = int(height_ft_match.group(2))
        result['height_cm'] = round(feet * 30.48 + inches * 2.54)

    # Weight in kg
    weight_kg_match = re.search(r'(\d{2,3})\s*kg', text, re.IGNORECASE)
    if weight_kg_match:
        result['weight_kg'] = int(weight_kg_match.group(1))

    # Weight in stone (9 st 14 lbs)
    weight_st_match = re.search(r'(\d+)\s*st\s*(\d+)\s*lbs', text, re.IGNORECASE)
    if weight_st_match:
        stone = int(weight_st_match.group(1))
        lbs = int(weight_st_match.group(2))
        total_lbs = stone * 14 + lbs
        result['weight_kg'] = round(total_lbs * 0.453592)

    return result

# =========================
# استخراج المهارات (بدون matching جزئي)
# =========================
def extract_skills(text):
    if not is_sports_cv(text):
        return {skill: 0 for skill in skill_keywords}

    text_lower = text.lower()
    skills = {}

    for skill, keywords in skill_keywords.items():
        found = 0
        for word in keywords:
            # matching بالكلمة الكاملة فقط
            if re.search(r'\b' + re.escape(word) + r'\b', text_lower):
                found = 1
                break
        skills[skill] = found

    return skills

# =========================
# استخراج المركز
# =========================
def extract_position(text):
    if not is_sports_cv(text):
        return "Not a Player"

    text_lower = text.lower()

    positions = {
        'ST': ['striker','forward','centre forward','center forward'],
        'RW': ['right wing'],
        'LW': ['left wing'],
        'CM': ['midfielder','central midfielder'],
        'CAM': ['attacking midfielder','playmaker'],
        'CDM': ['defensive midfielder'],
        'CB': ['center back','central defender'],
        'RB': ['right back'],
        'LB': ['left back'],
        'GK': ['goalkeeper','keeper']
    }

    for pos, keywords in positions.items():
        for word in keywords:
            if word in text_lower:
                return pos

    return "Unknown"

# =========================
# تحويل CV إلى DataFrame
# =========================
def process_cv(cv_text):
    achievements = count_achievements(cv_text)
    physical = extract_physical_info(cv_text)
    skills = extract_skills(cv_text)
    position = extract_position(cv_text)

    data = {**physical, **skills}
    data['main_position'] = position
    data['Achievements'] = achievements

    return pd.DataFrame([data])

In [ ]:
# =========================
# Install Docling
# =========================
!pip install docling

from google.colab import files
from docling.document_converter import DocumentConverter

# =========================
# Upload CV
# =========================
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

print("Uploaded:", file_name)

# =========================
# Read file using Docling
# =========================
converter = DocumentConverter()

result = converter.convert(file_name)

# استخراج النص
cv_text = result.document.export_to_text()

print("\nText length:", len(cv_text))
print("\nPreview:\n")
print(cv_text[:1000])

[INFO] 2026-05-06 14:12:56,205 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-06 14:12:56,207 [RapidOCR] device_config.py:57: Using CPU device


Saving football-cv-example2_ST.pdf to football-cv-example2_ST (2).pdf
Uploaded: football-cv-example2_ST (2).pdf


[INFO] 2026-05-06 14:12:56,292 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-06 14:12:56,295 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-06 14:12:56,739 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-06 14:12:56,741 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-05-06 14:12:56,749 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-05-06 14:12:56,751 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-05-06 14:12:56,917 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-06 14:12:56,918 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-05-06 14:12:57,075 [Ra

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]


Text length: 3364

Preview:

Mohamed Salah

by CV Genius

P E R S O N A L   P R O F I L E

D edicated, physically fit, and creative footballer committed to helping the team reach its season's objectives, winning trophies, and upholding the team's historical legacy. Demonstrate commitment to rigorous training regimes, resulting in a career-high of 13 goals last season. Leverage soft skills like empathy and active listening in coaching young academy team substitutes and fringe players aiming for a permanent place in their team's first eleven.

A T T R I B U T E S

- o Height: 5'9'
- o Weight: 9 st 14 lbs
- o 100 m speed: 11.8 seconds
- o Vertical: 23.6'

C A R E E R   S T A T I S T I C S

- o Goals: 31
- o Shots: 104
- o Assists: 43
- o Pass completion: 84.6%
- o PK: 10

A W A R D S

EFL League Two MVP

Poole Towne Best Player of the Month (3x)

P L A Y I N G   E X P E R I E N C E

Midfielder | December 2019 -P resent

NEWPORT COUNTY, Newport, Wales

- Created 59 big chances in debut se

In [ ]:
df = process_cv(cv_text)
df

,age,height_cm,weight_kg,passing,shooting,dribbling,defending,pace,movement_acceleration,movement_sprint_speed,...,power_strength,power_stamina,power_jumping,mentality_aggression,mentality_positioning,mentality_vision,teamwork,tactics,main_position,Achievements
0,None,175,64,1,1,1,0,1,0,0,...,0,0,0,0,1,0,1,1,ST,2


In [ ]:
!pip install pdfplumber python-docx spacy
!python -m spacy download en_core_web_sm
!pip install python-docx PyPDF2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 94.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from google.colab import files
import pdfplumber
import docx
import io

In [ ]:
# رفع ملف متطلبات النادي
uploaded_req = files.upload()
req_file_name = list(uploaded_req.keys())[0]
req_file_bytes = uploaded_req[req_file_name]

# قراءة الـPDF
def read_req_pdf(file_bytes):
    text = ""
    with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

if req_file_name.endswith(".pdf"):
    req_text = read_req_pdf(req_file_bytes)
else:
    raise ValueError("Unsupported file type for club requirements")

print("Club Requirements Preview:\n")
print(req_text[:1000])

Saving Club Player Requirements_CB.pdf to Club Player Requirements_CB (1).pdf
Club Requirements Preview:

Club Player Requirements – Official Document
Position Required: Center Back (CB)
General Profile
• Age: 26–32 years
• Height: 188–198 cm
• Weight: 82–95 kg
• Minimum Experience: 10 years
• Achievements: At least 5 major career accomplishments
Technical Skills
• Passing: Moderate
• Shooting: Low
• Dribbling: Low
• Defending: Excellent
• Pace: Moderate
• Movement Acceleration: Moderate
• Sprint Speed: Moderate
• Physical Strength: Very Strong
• Stamina: High
• Jumping Ability: Excellent
Mental & Tactical Attributes
• Aggression: Very High
• Positioning: Excellent
• Vision: Moderate
• Teamwork: Strong
• Tactical Awareness: Elite Defensive Awareness



In [ ]:
import re

def extract_club_requirements(text):
    text_lower = text.lower()
    req = {}

    # Technical Skills
    skills = ['passing','shooting','dribbling','defending','pace',
              'movement_acceleration','movement_sprint_speed','physic',
              'power_strength','power_stamina','power_jumping',
              'mentality_aggression','mentality_positioning','mentality_vision',
              'teamwork','tactics']

    for skill in skills:
        req[skill] = 1 if skill in text_lower else 0

    # Age
    age_match = re.search(r'age.*?(\d+)[\s\-]*(\d+)?', text_lower)
    if age_match:
        if age_match.group(2):
            req['age_min'] = int(age_match.group(1))
            req['age_max'] = int(age_match.group(2))
        else:
            req['age_min'] = int(age_match.group(1))
            req['age_max'] = None
    else:
        req['age_min'] = req['age_max'] = None

    # Height
    height_match = re.search(r'height.*?(\d{2,3})\s*cm', text_lower)
    req['height_cm'] = int(height_match.group(1)) if height_match else 0

    # Weight
    weight_match = re.search(r'weight.*?(\d{2,3})', text_lower)
    req['weight_kg'] = int(weight_match.group(1)) if weight_match else 0

    # Experience
    exp_match = re.search(r'(\d+)\s*years', text_lower)
    req['experience_years'] = int(exp_match.group(1)) if exp_match else 0

    # Achievements / Trophies
    trophy_match = re.search(r'(\d+)\s*(league|cup|title|championship|award|trophy)', text_lower)
    req['achievements'] = int(trophy_match.group(1)) if trophy_match else 0

    # Position mapping: full name -> abbreviation
    position_priority = [
    ('central attacking midfielder', 'CAM'),
    ('attacking midfielder', 'CAM'),
    ('playmaker', 'CAM'),
    ('defensive midfielder', 'CDM'),
    ('holding midfielder', 'CDM'),
    ('central midfielder', 'CM'),
    ('midfielder', 'CM'),
    ('striker', 'ST'),
    ('center forward', 'ST'),
    ('forward', 'ST'),
    ('right winger', 'RW'),
    ('right wing', 'RW'),
    ('left winger', 'LW'),
    ('left wing', 'LW'),
    ('center back', 'CB'),
    ('central defender', 'CB'),
    ('right back', 'RB'),
    ('left back', 'LB'),
    ('right midfielder', 'RM'),
    ('left midfielder', 'LM'),
    ('goalkeeper', 'GK'),
    ('keeper', 'GK'),
    ('st', 'ST'), ('rw', 'RW'), ('lw', 'LW'), ('cm', 'CM'), ('cam', 'CAM'),
    ('cdm', 'CDM'), ('cb', 'CB'), ('rb', 'RB'), ('lb', 'LB'), ('rm', 'RM'),
    ('lm', 'LM'), ('gk', 'GK')
]

    req['main_position'] = 'Unknown'
    for key, abbrev in position_priority: # Changed .items() to direct iteration
        if key in text_lower:
            req['main_position'] = abbrev
            break

    return req

In [ ]:
# استدعاء دالة استخراج متطلبات النادي
club_requirements = extract_club_requirements(req_text)

# عرض النتيجة
print("Extracted Club Requirements:\n", club_requirements)

Extracted Club Requirements:
 {'passing': 1, 'shooting': 1, 'dribbling': 1, 'defending': 1, 'pace': 1, 'movement_acceleration': 0, 'movement_sprint_speed': 0, 'physic': 1, 'power_strength': 0, 'power_stamina': 0, 'power_jumping': 0, 'mentality_aggression': 0, 'mentality_positioning': 0, 'mentality_vision': 0, 'teamwork': 1, 'tactics': 0, 'age_min': 26, 'age_max': None, 'height_cm': 198, 'weight_kg': 82, 'experience_years': 32, 'achievements': 0, 'main_position': 'CB'}


In [ ]:
import numpy as np

# تنظيف الـ requirements من القيم None
def clean_requirements(requirements):
    return {k: v for k, v in requirements.items() if v is not None}

# تحديد المهارات المستوفاة فقط
def get_met_skills(player, requirements):
    met_skills = []
    player_row = player.iloc[0]

    for key, req_value in requirements.items():

        # --- AGE ---
        if key == 'age_min':
            if player_row.get('age') is not None and player_row['age'] >= req_value:
                if 'age_max' not in requirements or player_row['age'] <= requirements.get('age_max', float('inf')):
                    met_skills.append('age')

        elif key == 'age_max':
            if 'age_min' not in requirements:
                if player_row.get('age') is not None and player_row['age'] <= req_value:
                    met_skills.append('age')

        # --- HEIGHT ---
        elif key == 'height_cm':
            if player_row.get('height_cm') is not None and player_row['height_cm'] >= req_value:
                met_skills.append('height_cm')

        # --- WEIGHT ---
        elif key == 'weight_kg':
            if player_row.get('weight_kg') is not None and player_row['weight_kg'] >= req_value:
                met_skills.append('weight_kg')

        # --- ACHIEVEMENTS ---
        elif key == 'achievements':
            if player_row.get('Achievements') is not None and player_row['Achievements'] >= req_value:
                met_skills.append('achievements')

        # --- BINARY SKILLS ---
        elif key in player_row:
            if req_value == 1 and player_row[key] == 1:
                met_skills.append(key)

    return list(set(met_skills))


# حساب التقييم
def evaluate_player(player, position_weights, requirements):

    # تنظيف البيانات
    requirements = clean_requirements(requirements)

    required_pos = requirements.get('main_position')

    if required_pos is None or required_pos == "Unknown":
        print("Warning: Club did not specify a valid position")
        return 0

    weights = position_weights.get(required_pos)

    if weights is None:
        print(f"Warning: No weights found for position: {required_pos}")
        return 0

    met_skills = get_met_skills(player, requirements)

    score = 0
    for skill_name in met_skills:
        if skill_name in weights:
            score += weights[skill_name]

    return score


# =========================
# الاستخدام
# =========================

score = evaluate_player(df, position_weights, club_requirements)

player_position = df['main_position'].iloc[0]
required_position = club_requirements.get('main_position', 'Unknown')

print(f"Player position: {player_position}")
print(f"Club required position: {required_position}")
print(f"Player suitability for {required_position}: {score:.4f}")

Player position: ST
Club required position: CB
Player suitability for CB: 0.0109
